# Assignment 4: Cognitive FAQ System Using Pandas

**UCS420 : Cognitive Computing**  
**Name:** Harshpreet Singh  
**Roll Number:** 1024170162  
**Branch:** Computer Science and Engineering

## Question 1

In [1]:
import pandas as pd
import re

roll_number_str = "1024170162"
categories = ["billing", "account", "general"]
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings and choose Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay by UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]
dynamic_entries = []
for digit in roll_number_str[-2:]:
    category = categories[int(digit) % 3]
    if category == "billing":
        entry = {"question": "where can i download my fee receipt",
                 "answer": "Download your receipt from the billing history page.",
                 "keywords": "receipt download payment", "category": category}
    elif category == "account":
        entry = {"question": "how can i change my registered phone",
                 "answer": "Update the phone number in account settings.",
                 "keywords": "phone update account", "category": category}
    else:
        entry = {"question": "where can i check service announcements",
                 "answer": "View the latest notices on the support home page.",
                 "keywords": "notices announcements updates", "category": category}
    dynamic_entries.append(entry)
df_faq = pd.DataFrame(fixed_entries + dynamic_entries)
display(df_faq)

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings and choose Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay by UPI, card, or net banking.",pay payment upi fee,billing
4,where can i download my fee receipt,Download your receipt from the billing history...,receipt download payment,billing
5,where can i check service announcements,View the latest notices on the support home page.,notices announcements updates,general


## Question 2

In [2]:
def score_faq_entries(query, df):
    query_words = set(re.findall(r"[a-z0-9]+", query.lower()))
    results = []
    for index, row in df.iterrows():
        text = " ".join(str(row[col]) for col in ["question", "answer", "keywords"]).lower()
        words = set(re.findall(r"[a-z0-9]+", text))
        score = len(query_words & words)
        results.append({"question": row["question"], "answer": row["answer"], "category": row["category"], "confidence_score": score})
    return pd.DataFrame(results).sort_values("confidence_score", ascending=False).reset_index(drop=True)

query = input("Enter a question: ")
display(score_faq_entries(query, df_faq))

,question,answer,category,confidence_score
0,what is the annual fee,The annual fee is Rs 500.,billing,2
1,what are your working hours,We are open 9 AM to 5 PM.,general,2
2,where can i download my fee receipt,Download your receipt from the billing history...,billing,1
3,how to reset password,Go to Settings and choose Reset Password.,account,0
4,how can i pay the fee,"You can pay by UPI, card, or net banking.",billing,0
5,where can i check service announcements,View the latest notices on the support home page.,general,0


## Question 3

In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

chosen_category = dynamic_entries[0]["category"]
display(same_category(chosen_category, df_faq))

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay by UPI, card, or net banking.",pay payment upi fee,billing
4,where can i download my fee receipt,Download your receipt from the billing history...,receipt download payment,billing


## Question 4

In [4]:
entry_index = 0
new_keyword = input("Enter a keyword: ").strip()
if new_keyword:
    keywords = df_faq.at[entry_index, "keywords"].split()
    if new_keyword.lower() not in [word.lower() for word in keywords]:
        keywords.append(new_keyword)
    df_faq.at[entry_index, "keywords"] = " ".join(keywords)
df_faq.to_csv(roll_number_str + "_faq_data.csv", index=False)
display(df_faq)

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge Cognitive Computing,billing
1,how to reset password,Go to Settings and choose Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay by UPI, card, or net banking.",pay payment upi fee,billing
4,where can i download my fee receipt,Download your receipt from the billing history...,receipt download payment,billing
5,where can i check service announcements,View the latest notices on the support home page.,notices announcements updates,general


## Question 5

In [5]:
print(df_faq.groupby("category").size())

category
account    1
billing    3
general    2
dtype: int64


## Question 6

In [6]:
def show_best_matches(query, df):
    ranked = score_faq_entries(query, df)
    if ranked.empty:
        print("No entries found")
        return
    best_score = ranked["confidence_score"].max()
    best_matches = ranked[ranked["confidence_score"] == best_score]
    if best_score == 0:
        print("No matching entries found")
    else:
        display(best_matches)

show_best_matches("fee", df_faq)
show_best_matches("reset password", df_faq)

,question,answer,category,confidence_score
0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,how can i pay the fee,"You can pay by UPI, card, or net banking.",billing,1
2,where can i download my fee receipt,Download your receipt from the billing history...,billing,1


,question,answer,category,confidence_score
0,how to reset password,Go to Settings and choose Reset Password.,account,2
